<a href="https://colab.research.google.com/github/citylearn-project/CityLearn/blob/master/examples/controller_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Controller examples

For a first complete v3 simulation and operational BAU, start with the [quickstart](https://www.citylearn.net/quickstart.html). This notebook retains the flat-interface controller, RL-wrapper and dataset-generation examples. Training runs need more time and dependencies than the first simulation.

The examples below use `citylearn_challenge_2023_phase_2_local_evaluation`, whose buildings use LSTM thermal dynamics, and the bundled SAC and MARLISA controllers. Both need PyTorch, which the `rl` extra installs. Install CityLearn with it in the notebook environment:

In [ ]:
%pip install "citylearn[rl]"

Every example simulates the first week of the dataset (168 hourly time steps), so that the notebook runs in a few minutes. Remove `simulation_start_time_step` and `simulation_end_time_step` to simulate the whole dataset, and train for many more episodes for meaningful results.

In [ ]:
WEEK = 24*7 - 1  # last time step of the first week

## CityLearn Control Agents

### Passive no-control agent

`NoControlAgent` (called `BaselineAgent` up to CityLearn 3.6.0) disables the active actions; it is not the operational [BusinessAsUsualAgent](https://www.citylearn.net/guides/baselines.html). Construct a fresh environment before changing controller type.

Run the following to simulate an environment where the storage systems and heat pumps are not controlled. The storage actions prescribed will be 0.0 and the heat pump will have no action, i.e. `None`, causing it to deliver the ideal load in the building time series files:

Because this agent disables the active actions, its environment no longer matches the configuration from which `evaluate_v2()` rebuilds the native BAU reference, so the KPIs are evaluated without that reference (`include_business_as_usual=False`).

In [ ]:
from citylearn.agents.base import NoControlAgent as Agent
from citylearn.citylearn import CityLearnEnv

# initialize
env = CityLearnEnv('citylearn_challenge_2023_phase_2_local_evaluation', central_agent=True, simulation_start_time_step=0, simulation_end_time_step=WEEK)
model = Agent(env)

# step through environment and apply agent actions
observations, _ = env.reset()

while not env.terminated:
    actions = model.predict(observations)
    observations, reward, terminated, truncated, info = env.step(actions)

# test
kpis = model.env.evaluate_v2(include_business_as_usual=False)
kpis = kpis.pivot(index='cost_function', columns='name', values='value').round(3)
kpis = kpis.dropna(how='all')
display(kpis)

### Centralized RBC
Run the following to simulate an environment controlled by centralized RBC agent for a single episode:

In [ ]:
from citylearn.agents.rbc import BasicRBC as Agent
from citylearn.citylearn import CityLearnEnv

# initialize
env = CityLearnEnv('citylearn_challenge_2023_phase_2_local_evaluation', central_agent=True, simulation_start_time_step=0, simulation_end_time_step=WEEK)
model = Agent(env)

# step through environment and apply agent actions
observations, _ = env.reset()

while not env.terminated:
    actions = model.predict(observations)
    observations, reward, terminated, truncated, info = env.step(actions)

# test
kpis = model.env.evaluate_v2()
kpis = kpis.pivot(index='cost_function', columns='name', values='value').round(3)
kpis = kpis.dropna(how='all')
display(kpis)

### Decentralized-Independent SAC

Run the following to simulate an environment controlled by decentralized-independent SAC agents for two episodes:

In [ ]:
from citylearn.agents.sac import SAC as Agent
from citylearn.citylearn import CityLearnEnv

# initialize
env = CityLearnEnv('citylearn_challenge_2023_phase_2_local_evaluation', central_agent=False, simulation_start_time_step=0, simulation_end_time_step=WEEK)
model = Agent(env)

# train
model.learn(episodes=2, deterministic_finish=True)

# test
kpis = model.env.evaluate_v2()
kpis = kpis.pivot(index='cost_function', columns='name', values='value').round(3)
kpis = kpis.dropna(how='all')
display(kpis)

### Decentralized-Cooperative MARLISA

Run the following to simulate an environment controlled by decentralized-cooperative MARLISA agents for two episodes:

In [ ]:
from citylearn.agents.marlisa import MARLISA as Agent
from citylearn.citylearn import CityLearnEnv

# initialize
env = CityLearnEnv('citylearn_challenge_2023_phase_2_local_evaluation', central_agent=False, simulation_start_time_step=0, simulation_end_time_step=WEEK)
model = Agent(env)

# train
model.learn(episodes=2, deterministic_finish=True)

# test
kpis = model.env.evaluate_v2()
kpis = kpis.pivot(index='cost_function', columns='name', values='value').round(3)
kpis = kpis.dropna(how='all')
display(kpis)

## Other Standard Reinforcement Learning Libraries

### Stable Baselines3 Reinforcement Learning Algorithms

Install Stable Baselines3 2.4 or newer, which supports gymnasium 1.x:

In [ ]:
%pip install "stable-baselines3>=2.4"

Before the environment is ready for use in Stable Baselines3, it needs to be wrapped. Firstly, wrap the environment using the `NormalizedObservationWrapper` (see [docs](https://www.citylearn.net/api/citylearn.wrappers.html#citylearn.wrappers.NormalizedObservationWrapper)) to ensure that observations served to the agent are min-max normalized between [0, 1] and cyclical observations e.g. hour, are encoded using the cosine transformation.

Next, we wrap with the `StableBaselines3Wrapper` (see [docs](https://www.citylearn.net/api/citylearn.wrappers.html#citylearn.wrappers.StableBaselines3Wrapper)) that ensures observations, actions and rewards are served in manner that is compatible with Stable Baselines3 interface.

> ⚠️ **NOTE**: `central_agent` in the `env` must be `True` when using Stable Baselines3  as it does not support multi-agents.

In [ ]:
from stable_baselines3.sac import SAC as Agent
from citylearn.citylearn import CityLearnEnv
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

# initialize
env = CityLearnEnv('citylearn_challenge_2023_phase_2_local_evaluation', central_agent=True, simulation_start_time_step=0, simulation_end_time_step=WEEK)
env = NormalizedObservationWrapper(env)
env = StableBaselines3Wrapper(env)
model = Agent('MlpPolicy', env)

# train
episodes = 2
model.learn(total_timesteps=env.unwrapped.time_steps*episodes)

# test
observations, _ = env.reset()

while not env.unwrapped.terminated:
    actions, _ = model.predict(observations, deterministic=True)
    observations, _, _, _, _ = env.step(actions)

kpis = env.unwrapped.evaluate_v2()
kpis = kpis.pivot(index='cost_function', columns='name', values='value').round(3)
kpis = kpis.dropna(how='all')
display(kpis)

### RLlib

CityLearn also provides `RLlibSingleAgentWrapper` and `RLlibMultiAgentEnv` (see the [wrappers API](https://www.citylearn.net/api/citylearn.wrappers.html)). Recent Ray releases pin their own gymnasium version and publish wheels only for some Python versions, and these wrappers are not tested against a current Ray release, so this notebook no longer runs an RLlib example. Install a Ray release that supports your Python version (`pip install "ray[rllib]"`) and wrap the environment with `NormalizedObservationWrapper` and `ClippedObservationWrapper` before the RLlib wrapper.

## Neighborhood Dataset Generation

Aside the provided datasets that come with the CityLearn installation, custom single-family residential datasets can be generated in CityLearn by taking advantage of the [End-Use Load Profiles for the U.S. Building Stock dataset](https://data.openei.org/submissions/4520). The [citylearn.end_use_load_profiles.neighborhood.Neighborhood](https://www.citylearn.net/api/citylearn.end_use_load_profiles.neighborhood.html#citylearn.end_use_load_profiles.neighborhood.Neighborhood) class makes this possible.

To learn more about the methodology used in this feature, refer to the [CityLearn v2 paper](https://doi.org/10.1080/19401493.2024.2418813).

Note that to make use of this feature, install `citylearn[end_use_load_profiles]` and [EnergyPlus 9.6.0](https://github.com/NREL/EnergyPlus/releases/tag/v9.6.0) must be installed. Other EnergyPlus versions are not yet supported.

An example of a generating a dataset and using it in simulation is:

In [ ]:
from citylearn.agents.rbc import BasicRBC as Agent
from citylearn.citylearn import CityLearnEnv
from citylearn.end_use_load_profiles.neighborhood import Neighborhood, SampleMethod

# path to version EnergyPlus 9.6.0 IDD
idd_filepath = '/Applications/EnergyPlus-9-6-0/PreProcess/IDFVersionUpdater/V9-6-0-Energy+.idd'

# build a neighborhood with n buildings through random sampling of single-family residential buildings in EULP dataset.
# Sampling population is filtered to include specific county and building vintage.
# train their LSTM thermal dynamics models and generate a CityLearn schema for the two buildings
neighborhood = Neighborhood()
n = 2
neighborhood_build = neighborhood.build(
    idd_filepath=idd_filepath,
    delete_energyplus_simulation_output=True,
    sample_buildings_kwargs=dict(
        sample_method=SampleMethod.RANDOM,
        sample_count=n,
        filters={
            'in.resstock_county_id': ['TX, Travis County'],
            'in.vintage': ['2000s']
        },
    ),
)

# simulate neighborhood in CityLearn
env = CityLearnEnv(neighborhood_build.schema_filepath, central_agent=True)
model = Agent(env)
observations, _ = env.reset()

while not env.terminated:
    actions = model.predict(observations)
    observations, reward, terminated, truncated, info = env.step(actions)

kpis = model.env.evaluate_v2()
kpis = kpis.pivot(index='cost_function', columns='name', values='value').round(3)
kpis = kpis.dropna(how='all')
display(kpis)